### ДЗ 2 — бейзлайны и оценка качества

**Датасет и постановка** - те же, что в `ДЗ1`: Breast Cancer Wisconsin (Diagnostic), бинарная классификация: по числовым признакам предсказать `diagnosis` (`M` / `B`). Столбец `id` в признаки не входит.

**Метрика**: основная - **recall по классу `M`** (не пропускать злокачественные случаи). Дополнительно для контроля показаны precision и F2 по `M` как и в разделе про метрику в первом ДЗ.


## 1) Импорты и воспроизводимость

Единый `RANDOM_STATE` для `train_test_split` и моделей с случайностью - чтобы при повторном запуске ноутбука результаты совпадали.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    recall_score,
    precision_score,
    fbeta_score,
    classification_report,
)

RANDOM_STATE = 42

print("OK")

OK


## 2) Загрузка данных

Используем тот же источник UCI и те же имена столбцов, что в ДЗ1: `DATA_URL`, `FEATURE_COLS`, `COLUMN_NAMES`.



In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"

FEATURE_COLS = [
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean",
    "smoothness_mean", "compactness_mean", "concavity_mean", "concave points_mean",
    "symmetry_mean", "fractal_dimension_mean",
    "radius_se", "texture_se", "perimeter_se", "area_se",
    "smoothness_se", "compactness_se", "concavity_se", "concave points_se",
    "symmetry_se", "fractal_dimension_se",
    "radius_worst", "texture_worst", "perimeter_worst", "area_worst",
    "smoothness_worst", "compactness_worst", "concavity_worst", "concave points_worst",
    "symmetry_worst", "fractal_dimension_worst",
]
COLUMN_NAMES = ["id", "diagnosis"] + FEATURE_COLS

df = pd.read_csv(DATA_URL, header=None, names=COLUMN_NAMES).dropna(axis=1, how="all")
print("Source:", DATA_URL, df.shape)
df.head()

Source: https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data (569, 32)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 3) Матрица признаков и таргет

- **`id`** - идентификатор, в модель не подаём.
- **`diagnosis`** - целевая переменная. Для sklearn удобно закодировать **бинарно**: `M` (злокачественная) => `1`, `B` => `0`. Тогда **recall по классу `M`** - это `recall_score(..., pos_label=1)`.

Отдельного категориального кодирования признаков не нужно: все 30 признаков в `X` числовые.


In [ ]:
target_col = "diagnosis"
id_col = "id" if "id" in df.columns else None

feature_cols = [c for c in df.columns if c not in {target_col, id_col}]
X = df[feature_cols].copy()

y = (df[target_col] == "M").astype(int)

print("X shape:", X.shape)
print("y: M=1, B=0  |  counts:\n", y.value_counts())
assert X.shape[1] == 30
assert set(y.unique()) == {0, 1}

X shape: (569, 30)
y: M=1, B=0  |  counts:
 diagnosis
0    357
1    212
Name: count, dtype: int64


## Вывод

**Размеры:** 569 объектов, 30 признаков - совпадает с EDA, id и diagnosis не входят в X.

**Таргет:** закодирован как B => 0, M => 1; классов два, всё ок для бинарной классификации.

**Баланс:** доброкачественных 357 (~ 62.7%), злокачественных 212 (~ 37.3%) - умеренный дисбаланс, поэтому разумно смотреть не только accuracy и опираться на recall(M), а split с stratify уместен.

## 4) Разбиение train / test

- **`test_size=0.2`** - отложенная выборка для честной оценки.
- **`stratify=y`** - сохраняем пропорции классов в train и test (в данных есть умеренный дисбаланс `B`/`M`).
- **`random_state=RANDOM_STATE`** - воспроизводимость.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("train:", X_train.shape[0], " test:", X_test.shape[0])
print("y_train:\n", y_train.value_counts(normalize=True).round(4))
print("y_test:\n", y_test.value_counts(normalize=True).round(4))

train: 455  test: 114
y_train:
 diagnosis
0    0.6264
1    0.3736
Name: proportion, dtype: float64
y_test:
 diagnosis
0    0.6316
1    0.3684
Name: proportion, dtype: float64


## Вывод

Доли классов на train и test близки, значит stratify отработал: тестовая выборка репрезентативна по балансу и сравнение моделей честное

## 5) Константный бейзлайн

`sklearn.dummy.DummyClassifier(strategy="most_frequent")` всегда предсказывает самый частый класс на **обучающей** выборке.

Ожидаемо, recall по `M` будет низким, если чаще встречается `B`: константа «всё B» не находит ни одного `M`.


In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

recall_m_dummy = recall_score(y_test, y_pred_dummy, pos_label=1)
precision_m_dummy = precision_score(y_test, y_pred_dummy, pos_label=1, zero_division=0)
f2_m_dummy = fbeta_score(y_test, y_pred_dummy, beta=2, pos_label=1, zero_division=0)

print("Most frequent label on train (1=M, 0=B):", int(y_train.mode().iloc[0]))
print(f"Recall(M):   {recall_m_dummy:.4f}")
print(f"Precision(M): {precision_m_dummy:.4f}")
print(f"F2(M):        {f2_m_dummy:.4f}")
print("\n", classification_report(y_test, y_pred_dummy, target_names=["B (0)", "M (1)"], zero_division=0))

Most frequent label on train (1=M, 0=B): 0
Recall(M):   0.0000
Precision(M): 0.0000
F2(M):        0.0000

               precision    recall  f1-score   support

       B (0)       0.63      1.00      0.77        72
       M (1)       0.00      0.00      0.00        42

    accuracy                           0.63       114
   macro avg       0.32      0.50      0.39       114
weighted avg       0.40      0.63      0.49       114



## Вывод

Константный бейзлайн повторяет наиболее частый класс (B), поэтому recall(M)=0: злокачественные случаи полностью пропускаются. Это показывает, что нужна модель, которая учится различать классы - дальше сравниваем с логрегом.

## 6) Бейзлайн-модель: логистическая регрессия + масштабирование

**Почему логрег:** относится к простому семейству линейных моделей, хорошо подходит для бинарной классификации.

**Почему `StandardScaler` в `Pipeline`:** в EDA видно, что масштабы признаков сильно различаются (например, площадь vs smoothness). Для логистической регрессии без масштабирования коэффициенты и оптимизация могут вести себя нестабильно; стандартная практика sklearn - `Pipeline` с `StandardScaler`, обученным **только на train** (при `fit` пайплайна на `X_train`).

**Параметры:** `max_iter=1000` - чтобы гарантировать сходимость; `random_state` - воспроизводимость.



In [ ]:
logreg_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                solver="lbfgs",
            ),
        ),
    ]
)

logreg_pipeline.fit(X_train, y_train)
y_pred_lr = logreg_pipeline.predict(X_test)

recall_m_lr = recall_score(y_test, y_pred_lr, pos_label=1)
precision_m_lr = precision_score(y_test, y_pred_lr, pos_label=1, zero_division=0)
f2_m_lr = fbeta_score(y_test, y_pred_lr, beta=2, pos_label=1, zero_division=0)

print(f"Recall(M):   {recall_m_lr:.4f}")
print(f"Precision(M): {precision_m_lr:.4f}")
print(f"F2(M):        {f2_m_lr:.4f}")
print("\n", classification_report(y_test, y_pred_lr, target_names=["B (0)", "M (1)"], zero_division=0))

Recall(M):   0.9286
Precision(M): 0.9750
F2(M):        0.9375

               precision    recall  f1-score   support

       B (0)       0.96      0.99      0.97        72
       M (1)       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



## Вывод

Логрег с масштабированием сильно лучше константного предсказания: recall(M) ≈ 0.93 - модель находит большую долю случаев с меткой M на отложенной выборке; precision(M) ≈ 0.98 - среди предсказанных M почти нет ложных срабатываний. F2(M) ≈ 0.94 отражает тот же упор на полноту по M, что и recall. Accuracy ≈ 0.96 показывает общую долю верных классов, но при дисбалансе классов её стоит читать вместе с recall/precision по M.

## 7) Сводка: качество на test

Основной ориентир по заданию - **Recall(M)** (полнота по классу злокачественных). Ниже сравнение константного бейзлайна и логрег + scaler на одной и той же отложенной выборке.


In [ ]:
summary = pd.DataFrame(
    {
        "model": ["Dummy (most_frequent)", "LogisticRegression + StandardScaler"],
        "recall_M": [recall_m_dummy, recall_m_lr],
        "precision_M": [precision_m_dummy, precision_m_lr],
        "F2_M": [f2_m_dummy, f2_m_lr],
    }
)
summary

,model,recall_M,precision_M,F2_M
0,Dummy (most_frequent),0.000000,0.000,0.0000
1,LogisticRegression + StandardScaler,0.928571,0.975,0.9375


## Вывод

На одной и той же отложенной выборке пайплайн с логрегом и масштабированием существенно превосходит константный бейзлайн по Recall(M) (и по связанным precision/F2 для M), то есть даёт осмысленное разделение классов вместо «угадывания большинства».